In [1]:
# import tensorflow
# from fast_soft_sort.tf_ops import soft_rank, soft_sort

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchsort  # pip install torchsort

def spearman_corr(pred, target, regularization_strength=1.0):
    # Compute differentiable soft ranks
    pred_ranks = torchsort.soft_rank(pred, regularization_strength=regularization_strength)
    target_ranks = torchsort.soft_rank(target, regularization_strength=regularization_strength)
    # Normalize to zero mean and unit norm
    pred_norm = (pred_ranks - pred_ranks.mean()) / pred_ranks.norm()
    target_norm = (target_ranks - target_ranks.mean()) / target_ranks.norm()
    print(pred_norm, target_norm)
    return (pred_norm * target_norm).sum()  # Cosine similarity ≈ correlation

# Example synthetic dataset
n_samples = 10
n_features = 3
torch.manual_seed(42)
X = torch.randn(n_samples, n_features)
true_weights = torch.tensor([0.2, -0.5, 1.0])
y = X @ true_weights + 0.1 * torch.randn(n_samples)
y = y.unsqueeze(0)

# Create linear model
model = nn.Linear(n_features, 1, bias=False)  # gives weights a_i
optimizer = optim.Adam(model.parameters(), lr=0.1)

for epoch in range(200):
    optimizer.zero_grad()
    y_hat = model(X).squeeze(0)
    corr = spearman_corr(y_hat, y, regularization_strength=0.5)
    loss = -corr  # we maximize correlation
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: Spearman ≈ {corr.item():.4f}")
        # print(model.weight.data)

print("Learned weights:", model.weight.data.squeeze())


tensor([[0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [3.7697e-08],
        [0.0000e+00]], grad_fn=<DivBackward0>) tensor([[ 0.0973, -0.1658,  0.0692, -0.0448, -0.1832,  0.0246, -0.0147,  0.1164,
         -0.1354,  0.2363]])
Epoch 0: Spearman ≈ 0.0000
tensor([[0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [3.7697e-08],
        [0.0000e+00]], grad_fn=<DivBackward0>) tensor([[ 0.0973, -0.1658,  0.0692, -0.0448, -0.1832,  0.0246, -0.0147,  0.1164,
         -0.1354,  0.2363]])
tensor([[0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [0.0000e+00],
        [3.7697e-08],
        [0.0000e+00]], grad_fn=<DivBackward0>) tensor([[ 0.0973, 

In [24]:
from scipy import stats
import numpy as np

y_hat = model(X).squeeze(0)
# corr = spearman_corr(y_hat, y, regularization_strength=0.5)

stats.spearmanr(y_hat.detach().numpy(), np.array(y[0]))

SignificanceResult(statistic=-0.9430303030303029, pvalue=1.2365316415492753e-48)

In [25]:
y_hat.detach().numpy()


array([[-3.75502199e-01],
       [ 1.26650286e+00],
       [-1.60581350e-01],
       [ 1.42782032e-01],
       [ 3.39680523e-01],
       [ 4.27706957e-01],
       [ 6.03546679e-01],
       [-7.61256337e-01],
       [-1.55584455e-01],
       [-3.98381323e-01],
       [ 9.70728040e-01],
       [-6.62639737e-01],
       [-3.49931598e-01],
       [-7.82082677e-01],
       [ 5.93277872e-01],
       [-6.57402992e-01],
       [-3.90213132e-02],
       [-8.24807048e-01],
       [ 5.94238937e-01],
       [-3.39288443e-01],
       [ 1.16041017e+00],
       [-2.44664088e-01],
       [-4.97370154e-01],
       [-8.81858706e-01],
       [ 1.81779206e-01],
       [ 3.82748187e-01],
       [ 2.57084548e-01],
       [-8.48415494e-02],
       [-4.78336304e-01],
       [ 4.48985457e-01],
       [ 1.22574508e+00],
       [ 3.96005392e-01],
       [ 3.69064003e-01],
       [-6.85766637e-02],
       [-3.03168207e-01],
       [-1.23664960e-01],
       [ 3.49732131e-01],
       [-3.52393955e-01],
       [-4.3